# Week 2 Day 1 Lab - Evals and Observability

## Measuring an agent on UnderCover's harder claims

A new batch of 25 claims has arrived at UnderCover, and they're harder than the queue from Week 1. Many have long lists of items, with payments from airlines, hotels and other insurers to take off, and deductibles, limits and caps to apply.

We trace the agent from Week 1 in Langfuse and score its decisions against the handlers'. Then we run two experiments: the agent working out each payout itself, and the agent with a payout tool.

## 4 parts:

PART 1: The harder claims

PART 2: Tracing the agent with Langfuse

PART 3: Scoring an agent

PART 4: Experiments in Langfuse

### My version of this lab

Parts 1 to 4 are Ed's, unchanged, up to the end of experiment 2. What I added, each with its conclusions:

- **Keeping the results**, after experiment 2: each experiment's table is saved to `week2day1_results.csv`, so the comparisons survive a kernel restart.
- **Experiment 3:** the agent with the payout tool on other models and reasoning levels, GPT-6 Luna and Grok 4.6 and 4.7.
- **Experiment 4:** the assignment, Ed's concise wording from Week 1 Day 4 against the full wording, and the cost of a wrongful decline scored again.
- **Experiment 5:** two other ways to shorten the wording that keep its meaning. One is a new file, `week2day1_policy_wording_short_v2.md`, with a note at the top on every change from Ed's version. Ed's file in `data/` is not changed.
- **What I found**, at the end.

In every experiment the instructions, the tools, the payout rules and the scoring are Ed's. Only the model, the reasoning level and the policy wording change. Experiment 3 calls Grok, which costs about 20 times more per token than Luna, so a full run of the notebook costs about $3.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls GPT-6 Luna through OpenRouter and costs a few cents. If you're OK to spend that, you'll need some credit on your account at <a href="https://openrouter.ai">openrouter.ai</a>. If you'd rather not spend anything, there's a free version of this lab in <a href="week2day1free.ipynb">week2day1free.ipynb</a>.<br/><br/>
            You'll also need a free Langfuse account, which we set up below. Each experiment takes 2 or 3 minutes to run. For a quicker run, set <code>TINY = True</code> in Part 4 to use 5 claims in place of 25.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import re
import sys
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler
# This notebook sits three folders below the repo's root, and Ed's payout rules are in labs/
sys.path.append("../../../labs")
from week2day1_payout import Benefit, Line, Payout, work_out_payout  # noqa: E402

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made in Week 1 in the `undercover` folder.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

## Connect to Langfuse

Langfuse is an open-source platform for tracing and evaluating LLM applications. You can host it yourself, or use Langfuse Cloud, which has a free Hobby plan.

1. Sign up at [cloud.langfuse.com](https://cloud.langfuse.com) and create a project
2. In the project's settings, go to API Keys and create a new key
3. Add the three values it shows to your `.env` file: `LANGFUSE_SECRET_KEY`, `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_BASE_URL`

In [ ]:
load_dotenv(override=True)
langfuse = get_client()

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("No Langfuse keys found. Add them to the .env file")

## Part 1: The harder claims

The harder claims are in `data/evals/harder`, with their own policy records and the handlers' decisions. The policy wording is the same as before.

In [ ]:
HARDER = Path("../../../data/evals/harder")


def load(name):
    """Read one of the JSON files of the harder claims."""
    return json.loads((HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../../../data/policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "items": len(claim["items"]),
        "amount_claimed": claim["amount_claimed"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

Similar mix of decisions

In [ ]:
data["decision"].value_counts()

### A burglary in Rome

Here's one of the longer claims.

In [ ]:
rome = [claim for claim in claims if claim["claim_id"] == "CLM-1203"][0]

print(rome["summary"])
print(rome["story"])
pd.DataFrame(rome["items"])

This is how the handler worked out the payout.

In [ ]:
decision = labels["CLM-1203"]

print(decision["reason"])
pd.DataFrame(decision["calculation"])

## Part 2: Tracing the agent with Langfuse

This is the agent from Week 1 Day 2 with two changes. The customer's policy record goes in the prompt with the claim, so its only tool is `record_decision`. We also turn Luna's reasoning off with `reasoning_effort="none"`, which makes each call faster and cheaper.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

The tool's docstring is in Google style. With `parse_docstring=True`, LangChain turns each line under `Args` into the description of that argument.

The rationale comes before the decision, so the model writes its reasons before it decides. As in UnderCover's claims system, a decision can't be changed once it's recorded, and an approval needs a payout above zero.

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [ ]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="none", timeout=120, max_retries=5)

baseline_agent = create_agent(model=luna, tools=[record_decision], system_prompt=f"{INSTRUCTIONS}\n{policy_wording}\n")

Langfuse's `CallbackHandler` sends a trace of everything LangChain does to Langfuse. We pass it to the agent in the `config`.

In [ ]:
langfuse_handler = CallbackHandler()

recorded.clear()
result = baseline_agent.invoke(
    {"messages": [{"role": "user", "content": make_user_prompt(rome)}]},
    config={"callbacks": [langfuse_handler]},
)

for message in result["messages"][1:]:
    message.pretty_print()

Now open **Tracing** in Langfuse. The trace shows each call to the model with its prompt, its reply, the tokens and the time it took, and the tool call in between.

## Part 3: Scoring an agent

In Week 1 a claim was right when the decision matched the handler's and the payout was within a dollar. We still count that, but not every mistake costs UnderCover the same.

### The confusion matrix

A confusion matrix compares an agent's decisions with the right answers. Each row is the handler's decision and each column is the agent's. The claims on the diagonal are the ones where they agree, and each cell off the diagonal is a kind of mistake:

- **A wrongful decline:** the agent declines a claim the handler approved or referred. The customer is let down and may complain.
- **A wrongful payment:** the agent approves a claim the handler declined or referred. UnderCover pays money it doesn't owe.
- **A needless referral:** the agent refers a claim the handler decided. A handler has to spend time on it.

The confusion matrix compares decisions only, so an approval with the wrong payout still sits on the diagonal. The cost of mistakes below counts the wrong amounts.

### Precision and recall

Precision and recall come from the same matrix, with an approval as the positive:

- **Precision:** of the claims the agent approved, the share the handler approved.
- **Recall:** of the claims the handler approved, the share the agent approved.

### The cost of mistakes

A cost matrix gives each kind of mistake a cost in dollars. A wrongful payment costs the amount paid, and a wrong amount on a correct approval costs the difference.

In [ ]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">The UnderCover business has more tolerance for false positives (we cover someone when we shouldn't) than false negatives (we decline someone when we should have covered them), because false negatives damage our reputation and customer loyalty. Our evals take this into account.<br/><br/>
            Agree the figures in the cost matrix with the business before you run your evals.
            </span>
        </td>
    </tr>
</table>

### Kappa

Accuracy can flatter an agent. 14 of the 25 claims are approvals, so an agent that approves everything gets 56% of the decisions right. Cohen's kappa corrects for this. It measures how much more often the agent agrees with the handlers than it would by chance, based on how often each of them picks each decision. A kappa of 1 is perfect agreement and 0 is no better than chance.

In [ ]:
def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


approve_everything = pd.DataFrame({"handler": data["decision"], "agent": "approve"})

print(f"Approve everything gets a kappa of {kappa(approve_everything):.2f}")

Here's the confusion matrix for approving everything. Every claim lands in the approve column.

In [ ]:
def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()


show_confusion_matrix(approve_everything, "Approve everything")

### The straight-through rate

The straight-through rate is the share of claims the agent decides without referring them to a handler. It includes approvals over $1,000, which still need a handler's sign-off. An agent that refers every claim never pays or declines a claim wrongly. But its straight-through rate is 0%, and each of its 19 needless referrals costs $50. The handlers refer 6 of the 25 claims, so their rate is 76%.

## Part 4: Experiments in Langfuse

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Datasets and experiments</h2>
            <span style="color:#118AB2;">A Langfuse dataset holds test inputs with their expected outputs. An experiment runs a task on each item of a dataset and scores the result with evaluators. An item evaluator scores one item and a run evaluator scores the whole run.<br/><br/>
            Langfuse keeps every run of a dataset with its scores and traces, so you can compare runs side by side.
            </span>
        </td>
    </tr>
</table>

### The dataset

We upload the harder claims as a dataset, with the handler's decision as the expected output. Each item's id is the claim id, so running this cell again updates the items in place.

In [ ]:
DATASET = "harder-claims"

langfuse.create_dataset(name=DATASET, description="UnderCover's harder claims with the handlers' decisions")

for claim in claims:
    label = labels[claim["claim_id"]]
    langfuse.create_dataset_item(
        dataset_name=DATASET,
        id=claim["claim_id"],
        input=claim,
        expected_output={"decision": label["decision"], "payout": label["payout"], "reason": label["reason"]},
        metadata={"tiny": label["tiny"]},
    )

By default we run all 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

`N_TIMES` is the number of times each experiment runs. The agent's answers vary from run to run, and more runs give steadier scores. With more than one run, each confusion matrix adds up the claims from every run, and the comparison at the end averages the scores.

In [ ]:
TINY = False
N_TIMES = 1

dataset = langfuse.get_dataset(DATASET)
items = dataset.items
if TINY:
    items = [item for item in items if item.metadata["tiny"]]

print(f"Running {len(items)} of the {len(dataset.items)} claims")

### The task and the evaluators

The task runs an agent on one claim. If the agent finishes without recording a decision, the claim would go to a handler, so we count it as a referral.

In [ ]:
def decide(agent, claim):
    """Run an agent on one claim and return its decision, with the cost of its model calls."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [langfuse_handler]},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}")
    return {**answer, "cost": cost}

`score_claim` is the item evaluator and `score_run` is the run evaluator. `score_table` puts each claim's answer beside the handler's.

In the run's scores, `decision_accuracy` is the share of claims on the diagonal of the confusion matrix. `accuracy` also needs the payout within a dollar.

In [ ]:
def score_claim(*, output, expected_output, **kwargs):
    """Item evaluator: whether the agent got the claim right, and the cost of its mistake."""
    return [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]


def score_table(item_results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "llm_cost": answer["cost"],
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "llm_cost": table["llm_cost"].sum(),
    }


def score_run(*, item_results, **kwargs):
    """Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]

### Experiment 1: the baseline

The agent works out each payout itself, following section 10 of the policy wording. Langfuse runs the claims one at a time, so each run takes about 2 minutes.

In [ ]:
def baseline_task(*, item, **kwargs):
    """Run the baseline agent on one claim from the dataset."""
    return decide(baseline_agent, item.input)


baseline_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    baseline = langfuse.run_experiment(
        name="Baseline",
        description="The agent works out each payout itself",
        data=items,
        task=baseline_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(baseline.format())
    baseline_tables.append(score_table(baseline.item_results).assign(run=run))

In [ ]:
baseline_table = pd.concat(baseline_tables, ignore_index=True)
show_confusion_matrix(baseline_table, "Baseline")

Here are the claims it got wrong, with the agent's rationale.

In [ ]:
baseline_table[~baseline_table["right"]]

### Experiment 2: a payout tool

The payout rules in section 10 are fixed business logic, and business logic belongs in code. `week2day1_payout.py` has UnderCover's payout rules as a Python function, `work_out_payout`, and we give it to the agent as a tool. The agent still judges whether the claim is covered, which lines count and what counts as one item.

In [ ]:
@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

A tool is still a Python function, so we can try it ourselves. Here's a suitcase and three T-shirts on an Economy policy, where the airline has paid $100.

In [ ]:
calculate_payout.invoke({
    "policy_id": "POL-2201",
    "benefit": "baggage",
    "lines": [
        {"description": "Hard-shell suitcase", "amount": 239.99, "kind": "item"},
        {"description": "Three T-shirts", "amount": 74.97, "quantity": 3, "kind": "item"},
    ],
    "paid_by_others": 100,
})

The agent gets the tool and one more line in its instructions.

In [ ]:
PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"

payout_agent = create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n")


def payout_tool_task(*, item, **kwargs):
    """Run the agent with the payout tool on one claim from the dataset."""
    return decide(payout_agent, item.input)

In [ ]:
payout_tool_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    with_payout_tool = langfuse.run_experiment(
        name="Payout tool",
        description="The agent works out each payout with the payout tool",
        data=items,
        task=payout_tool_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(with_payout_tool.format())
    payout_tool_tables.append(score_table(with_payout_tool.item_results).assign(run=run))

In [ ]:
payout_tool_table = pd.concat(payout_tool_tables, ignore_index=True)
show_confusion_matrix(payout_tool_table, "Payout tool")

In [ ]:
payout_tool_table[~payout_tool_table["right"]]

In [ ]:
print("The baseline got either decision or amount wrong for:", len(baseline_table[~baseline_table["right"]]) )
print("The payout tool version got either decision or amount wrong for:", len(payout_tool_table[~payout_tool_table["right"]]) )


### Keeping the results

Each experiment's table goes into `week2day1_results.csv` beside this notebook, one row per claim per run, so the comparison at the end survives a kernel restart and the model runs in experiment 3 can be added one at a time. An experiment already in the file is skipped.

In [ ]:
RESULTS = Path("week2day1_results.csv")


def saved_results():
    """The tables saved so far, one row per claim per run, with the experiment's name."""
    return pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame(columns=["experiment"])


def is_saved(name):
    """True when the results file holds this experiment for every claim and every run of the current TINY and N_TIMES."""
    return (saved_results()["experiment"] == name).sum() == len(items) * N_TIMES


def save_results(name, table):
    """Save an experiment's table to the results file, in place of any rows it had there before, unless it is already complete."""
    if is_saved(name):
        print(f"{name}: already in {RESULTS}")
        return
    saved = saved_results()
    pd.concat([saved[saved["experiment"] != name], table.assign(experiment=name)], ignore_index=True).to_csv(RESULTS, index=False)


save_results("Baseline, Luna, no reasoning", baseline_table)
save_results("Payout tool, Luna, no reasoning", payout_tool_table)

Open the traces for these claims in Langfuse to see the lines the agent passed to the tool and the calculation that came back.

### Experiment 3: other models and reasoning

The lab runs GPT-6 Luna with reasoning off. Experiment 2's agent, the payout tool and the prompt stay the same here, and only the model and its reasoning effort change: Luna at medium and high reasoning, Grok 4.6 at low, medium and high, and Grok 4.7 at low.

Grok costs about 20 times more per token than Luna on OpenRouter, so each run is about a dollar rather than a cent. Each model's table is appended to `week2day1_results.csv` beside this notebook as it finishes, so the comparison survives a kernel restart, and a model already in the file is skipped.

In [ ]:
MODELS = [
    ("Payout tool, Luna, medium reasoning", "openai/gpt-6-luna", "medium"),
    ("Payout tool, Luna, high reasoning", "openai/gpt-6-luna", "high"),
    ("Payout tool, Grok 4.6, low reasoning", "x-ai/grok-4.6", "low"),
    ("Payout tool, Grok 4.6, medium reasoning", "x-ai/grok-4.6", "medium"),
    ("Payout tool, Grok 4.6, high reasoning", "x-ai/grok-4.6", "high"),
    ("Payout tool, Grok 4.7, low reasoning", "x-ai/grok-4.7", "low"),
]


def make_payout_agent(model, wording=policy_wording):
    """Experiment 2's agent, with the payout tool, on any model and any version of the wording."""
    return create_agent(model=model, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{wording}\n")


def run_model(name, model_id, reasoning_effort, wording=policy_wording):
    """Run the agent with the payout tool on one model N_TIMES, as experiments in Langfuse, and return the table of every run, with the time it took.

    The wording is the text of the policy wording, or a function that gives the text for each claim.
    """
    model = ChatOpenAI(model=model_id, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort=reasoning_effort, timeout=300, max_retries=5)

    def agent_for(claim):
        """The agent for one claim, with the wording that claim gets."""
        return make_payout_agent(model, wording(claim) if callable(wording) else wording)

    tables = []
    for run in range(1, N_TIMES + 1):
        recorded.clear()
        start = time.time()
        result = langfuse.run_experiment(
            name=name,
            description=f"The agent works out each payout with the payout tool, on {model_id} with {reasoning_effort} reasoning",
            data=items,
            task=lambda *, item, **kwargs: decide(agent_for(item.input), item.input),
            evaluators=[score_claim],
            run_evaluators=[score_run],
            max_concurrency=1,
        )
        print(result.format())
        tables.append(score_table(result.item_results).assign(run=run, seconds_per_claim=(time.time() - start) / len(items)))
    return pd.concat(tables, ignore_index=True).assign(experiment=name)

In [ ]:
for name, model_id, reasoning_effort in MODELS:
    if is_saved(name):
        print(f"{name}: already in {RESULTS}")
        continue
    save_results(name, run_model(name, model_id, reasoning_effort))

### Comparing the experiments

The lab's scores for every experiment in `week2day1_results.csv`, averaged over the runs.

In [ ]:
def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()


order = ["Baseline, Luna, no reasoning", "Payout tool, Luna, no reasoning", *(name for name, _, _ in MODELS)]
tables = {name: table for name, table in saved_results().groupby("experiment")}
tables = {name: tables[name] for name in order if name in tables}
comparison = pd.DataFrame([average_scores(table) for table in tables.values()], index=list(tables)).round(3)
comparison

The claims each model got wrong in any run, with the agent's rationale.

In [ ]:
for name, table in tables.items():
    wrong = table[~table["right"]]
    print(f"{name}: {len(wrong)} wrong of {len(table)}")
    display(wrong[["run", "claim_id", "handler", "agent", "handler_payout", "agent_payout", "rationale"]])

### Conclusions

One run of each experiment on the 25 harder claims, on October 29, 2026:

| Experiment | Wrong of 25 | Accuracy | Recall | Cost of mistakes | Straight-through | Model cost per run | Time per run |
|---|---|---|---|---|---|---|---|
| Baseline, Luna, no reasoning | 14 | 0.44 | 0.57 | $9,775 | 0.60 | $0.008 | 2 min |
| Payout tool, Luna, no reasoning | 3 | 0.88 | 0.86 | $1,583 | 0.72 | $0.011 | 2 min |
| Payout tool, Luna, medium reasoning | 2 | 0.92 | 0.93 | $100 | 0.68 | $0.018 | 3 min |
| Payout tool, Luna, high reasoning | 1 | 0.96 | 1.00 | $50 | 0.72 | $0.019 | 6 min |
| Payout tool, Grok 4.6, low reasoning | 3 | 0.88 | 0.79 | $150 | 0.64 | $0.41 | 4 min |
| Payout tool, Grok 4.6, medium reasoning | 0 | 1.00 | 1.00 | $0 | 0.76 | $0.72 | 5 min |
| Payout tool, Grok 4.6, high reasoning | 0 | 1.00 | 1.00 | $0 | 0.76 | $0.85 | 20 min |
| Payout tool, Grok 4.7, low reasoning | 3 | 0.88 | 0.79 | $150 | 0.64 | $0.54 | 6 min |

**The payout tool is the biggest single gain, whatever the model.** It takes Luna from 14 wrong to 3, and every model after it is within 3 of perfect. The arithmetic of section 10 is business logic, and business logic in code beats any amount of reasoning about it.

**Reasoning is the cheapest next step.** Luna with medium reasoning costs 2 cents a run, under twice Luna without it, and cuts the cost of mistakes from $1,583 to $100. Luna with high reasoning costs the same 2 cents, takes twice as long, and gets 24 of 25 with every approval right. Every mistake Luna makes with reasoning on is a needless referral, the mildest kind, and the one it shares at medium and high is CLM-1213: a bag left on the beach, where the agent reads "could see the towels most of the time" as unsettled and the handler reads 40 meters away while swimming as out of reach. The errors Luna makes without reasoning are of the kind reasoning removes: a rationale that says the claim is covered, and a decision that says decline.

**Grok 4.6 at medium reasoning is the only perfect run in this experiment, and high buys nothing more.** Both score 25 of 25, and high takes four times as long and costs 20% more. Grok 4.7 at low reasoning is no better than 4.6 at low. Per claim, Grok medium costs about 3 cents against Luna high's 0.08 cents, nearly 40 times more, and on this run buys one needless referral, $50 on UnderCover's cost matrix. Luna at high reasoning is the best value here: one mild mistake for 2 cents a run.

**Every error on low reasoning is the same error, and it is in our instructions.** All six wrong answers from Grok 4.6 low and Grok 4.7 low are approvals the handler made, where the agent worked out the right payout to the cent and then referred the claim because the tool returned `needs_sign_off: true`. The rationale says so each time: "the payout calculation needs sign-off, so a colleague must review". Nothing in the instructions says that sign-off is a handler's signature on an approval over $1,000, not a reason to refer. One line would fix it, and would likely bring low reasoning level with medium at half the cost. That is the next experiment to run, before buying a bigger model.

**The straight-through rate shows the same thing.** The handlers decide 76% of these claims without a referral. Grok medium and high match that exactly. The low-reasoning runs sit at 64% because of the sign-off referrals, and each needless referral costs a handler's time.

**One run is not a measurement.** The same configuration, Luna without reasoning on the full wording with the payout tool, runs twice in this notebook, in experiments 2 and 4, with 3 and 4 wrong, and on other runs the same day it had anywhere from 0 to 5. A difference of one or two mistakes between the rows above is noise. For a decision that costs money, run each candidate three times (`N_TIMES = 3`) and compare the averages, as the lab suggests.

In Langfuse, open **Datasets** and then `harder-claims`. Select the runs to compare them side by side, claim by claim.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: on Week 1 Day 4 we wrote a concise version of the policy wording, in <code>data/policy_wording_short.md</code>. Run a third experiment with the concise wording and the payout tool, and compare it with the full wording in Langfuse. Does it keep the accuracy? Is it cheaper or faster?<br/><br/>
            Then change <code>WRONGFUL_DECLINE</code> and score the runs again. Does it change which wording you would choose?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

### Experiment 4: the concise wording

The assignment above. On Week 1 Day 4 we wrote a concise version of the policy wording, in `data/policy_wording_short.md`, with the same rules in about 60% of the words. The agent with the payout tool runs with each wording, on Luna without reasoning, as in experiment 2, and on Luna with medium and high reasoning, the best value in experiment 3.

All of them run here, the full wording too, so each is timed the same way. The runs land in Langfuse under the `harder-claims` dataset, named after the wording and the model.

In [ ]:
short_wording = Path("../../../data/policy_wording_short.md").read_text(encoding="utf-8")
print(f"Full wording: {len(policy_wording.split()):,} words. Concise wording: {len(short_wording.split()):,} words")

WORDING_RUNS = [
    ("Full wording, Luna, no reasoning", "openai/gpt-6-luna", "none", policy_wording),
    ("Concise wording, Luna, no reasoning", "openai/gpt-6-luna", "none", short_wording),
    ("Full wording, Luna, medium reasoning", "openai/gpt-6-luna", "medium", policy_wording),
    ("Concise wording, Luna, medium reasoning", "openai/gpt-6-luna", "medium", short_wording),
    ("Full wording, Luna, high reasoning", "openai/gpt-6-luna", "high", policy_wording),
    ("Concise wording, Luna, high reasoning", "openai/gpt-6-luna", "high", short_wording),
]

In [ ]:
for name, model_id, reasoning_effort, wording in WORDING_RUNS:
    if is_saved(name):
        print(f"{name}: already in {RESULTS}")
        continue
    save_results(name, run_model(name, model_id, reasoning_effort, wording))

The lab's scores for each wording, with the time per claim. In Langfuse, open **Datasets**, then `harder-claims`, and select the two runs on the same model to compare the wordings claim by claim.

In [ ]:
wording_names = [name for name, *_ in WORDING_RUNS]
wording_tables = {name: table for name, table in saved_results().groupby("experiment") if name in wording_names}
wording_tables = {name: wording_tables[name] for name in wording_names if name in wording_tables}

wording_comparison = pd.DataFrame([average_scores(table) for table in wording_tables.values()], index=list(wording_tables))
wording_comparison["seconds_per_claim"] = [table["seconds_per_claim"].mean() for table in wording_tables.values()]
wording_comparison.round(3)

In [ ]:
for name, table in wording_tables.items():
    wrong = table[~table["right"]]
    print(f"{name}: {len(wrong)} wrong of {len(table)}")
    display(wrong[["run", "claim_id", "handler", "agent", "handler_payout", "agent_payout", "rationale"]])

### Changing the cost of a wrongful decline

`WRONGFUL_DECLINE` is what the business says a wrongful decline costs UnderCover, in complaints and lost customers. The cost of mistakes scores the runs again for each value, with no calls to the model.

In [ ]:
costs = {}
for WRONGFUL_DECLINE in (500, 1500, 5000, 15000):
    costs[f"${WRONGFUL_DECLINE:,}"] = {
        name: table.apply(
            lambda row: mistake_cost({"decision": row["agent"], "payout": row["agent_payout"]}, {"decision": row["handler"], "payout": row["handler_payout"]}),
            axis=1,
        ).groupby(table["run"]).sum().mean()
        for name, table in wording_tables.items()
    }
WRONGFUL_DECLINE = 1500

pd.DataFrame(costs).round(0).rename_axis(columns="Cost of a wrongful decline")

### Conclusions on the wording

One run of each, on the 25 harder claims, on October 29, 2026:

| Wording and model | Wrong of 25 | Accuracy | Cost of mistakes | Model cost per run | Seconds per claim |
|---|---|---|---|---|---|
| Full, Luna, no reasoning | 4 | 0.84 | $1,767 | $0.011 | 6.3 |
| Concise, Luna, no reasoning | 4 | 0.84 | $1,867 | $0.010 | 5.8 |
| Full, Luna, medium reasoning | 1 | 0.96 | $50 | $0.017 | 15.7 |
| Concise, Luna, medium reasoning | 4 | 0.84 | $1,928 | $0.017 | 14.0 |
| Full, Luna, high reasoning | 2 | 0.92 | $350 | $0.017 | 12.8 |
| Concise, Luna, high reasoning | 5 | 0.80 | $2,214 | $0.021 | 17.2 |

**Without reasoning, the concise wording keeps the accuracy and saves little.** Both wordings get 21 of 25. The concise one has 41% fewer words, but the run costs only 8% less and is 8% faster, because the wording is only part of each prompt: the policy record, the claim with its documents, and the tool calls are the rest. Both runs get CLM-1207 and CLM-1209 wrong, and their other mistakes differ, which is the noise of one run.

**With reasoning, the concise wording is worse at both levels.** At medium it gets 4 wrong against 1, at high 5 against 2, and its mistakes cost $1,900 to $2,200 a run against $50 to $350. It saves nothing either: the model reasons about as long over the shorter text, so the cost per run is the same or higher. The seconds per claim move by a few seconds from run to run on OpenRouter, so read them as rough.

**One mistake is the concise wording's own, and it repeats.** The full wording says scuba diving "deeper than 10 meters" is an adventure sport. The concise one says "scuba below 10 meters", which can be read either way. On CLM-1217, a dive to 9 meters, Luna at both medium and high reasoning read 9 meters as below 10 and declined the claim, and at medium the rationale quotes the phrase word for word. Neither run on the full wording makes that mistake. On CLM-1209, grade 3 rafting, the concise wording also left Luna at medium unsure whether the add-on is needed, and it referred a claim the handler approved. A shorter wording has to keep the defining words exact: every threshold, every "and" in "out of sight and out of reach".

**Changing `WRONGFUL_DECLINE` doesn't change the choice.** The full wording with medium or high reasoning has no wrongful decline, so its cost of mistakes stays at $50 or $350 whatever a wrongful decline costs. Every other run has exactly one, so its cost rises dollar for dollar with `WRONGFUL_DECLINE`, from $500 to $15,000. Between the two wordings without reasoning the gap stays at $100 at every value. The higher the business puts the cost of a wrongful decline, the more the full wording with reasoning is worth, and at no value does the concise wording come out ahead.

**I would keep the full wording, with Luna at medium reasoning.** Full wording with medium reasoning is the best run here: 24 of 25 for under 2 cents. The concise wording saves about a tenth of a cent per run without reasoning and nothing with it, and its shortcut in the words cost a wrongful decline at both reasoning levels.

**The one mistake that every Luna run with reasoning shares is CLM-1213.** Luna at medium and high reasoning has now run four times on the full wording across experiments 3 and 4, and every run refers the bag left on the beach while the customer swam 40 meters away, because she says she could see the towels most of the time. That is a stable disagreement with the handler, not noise, and the place to settle it is the wording or the instructions, not the model.

### Experiment 5: shortening the wording without losing its meaning

Experiment 4 used Ed's concise wording from Week 1 Day 4, `data/policy_wording_short.md`, as it is. It rewrites the rules in fewer words, and with reasoning on it did worse than the full wording. The agent's rationales show why: some of its shortcuts change what a rule means.

- "scuba below 10 meters", for the full wording's "deeper than 10 meters", led to a wrongful decline on CLM-1217, a dive to 9 meters, at both medium and high reasoning.
- "Adventure Sports covers …", for "Adventure sports are …", left the agent unsure whether grade 3 rafting is excluded, on CLM-1209.
- "the value of data or work on a device" leaves out a paper notebook, which matters on CLM-1207, a sketchbook of drawings.
- The instructions say "work out the payout using section 10", and the concise wording has no section numbers.

Week 1 Day 4 shortened the prompt in three ways: a concise wording, only the sections each claim needs, and the claim as compact JSON without the text of its documents. Experiment 4 tried the first. This experiment tries the first again, done differently, and the second:

1. **A new concise wording**, in a new file beside this notebook, `week2day1_policy_wording_short_v2.md`. Ed's file in `data/` is left as it is. The new file shortens only the prose, and copies definitions, thresholds, lists and examples word for word from the full wording. The comment at the top of the file lists every change from Ed's version, and is taken out before the wording goes into the prompt.
2. **Only the sections a claim needs, from the full wording**, with no rewording at all: its benefit's section, and the sections every claim needs. Week 1 Day 4 gave the add-ons section only when the policy had an add-on, but section 8 is also where adventure sports and electronics are defined, and claims without an add-on need those definitions (CLM-1209, CLM-1217). So section 8 always goes in.
3. **Both**: only the sections a claim needs, from the new concise wording.

The third way, leaving out the text of the documents, isn't tried: on these claims the decision turns on what the documents say.

What changes from Ed's lab is the policy wording in the system prompt and nothing else. The instructions, the tools, the payout rules and the scoring are his. All three run on Luna with medium reasoning, the best run in experiment 4, once each, and experiment 4's runs of the full wording and Ed's concise wording on the same model are the comparison.

In [ ]:
def without_comments(text):
    """The text with any HTML comment taken out, so the note at the top of a file stays out of the prompt."""
    return re.sub(r"<!--.*?-->\s*", "", text, flags=re.DOTALL)


short_wording_v2 = without_comments(Path("week2day1_policy_wording_short_v2.md").read_text(encoding="utf-8"))

for name, wording in [("Full wording", policy_wording), ("Ed's concise wording", short_wording), ("New concise wording", short_wording_v2)]:
    print(f"{name}: {len(wording.split()):,} words")

Both the full wording and the new concise one number their sections 1 to 12, so a claim's sections can be picked by number. Every claim gets the plan and its limits (1), the add-ons and their definitions (8), the general exclusions (9), how the payment is worked out (10), how to claim (11) and the definitions (12), and then the section for its own benefit.

In [ ]:
BENEFIT_SECTION = {
    "trip_cancellation": "2",
    "trip_interruption": "2",
    "trip_delay": "3",
    "baggage_delay": "4",
    "baggage": "5",
    "passport": "6",
    "emergency_medical": "7",
}
EVERY_CLAIM_SECTIONS = ["1", "8", "9", "10", "11", "12"]


def split_numbered_sections(wording):
    """The opening of the wording, and its sections keyed by their number."""
    opening, *parts = wording.split("\n## ")
    return opening, {part.split(".")[0]: "## " + part for part in parts}


def sections_for(wording):
    """A function that gives each claim the opening of the wording and only the sections that claim needs."""
    opening, sections = split_numbered_sections(wording)

    def wording_for(claim):
        wanted = sorted({BENEFIT_SECTION[claim["claim_type"]], *EVERY_CLAIM_SECTIONS}, key=int)
        return "\n".join([opening.rstrip() + "\n", *(sections[number].rstrip() + "\n" for number in wanted)])

    return wording_for


SHORTENING_RUNS = [
    ("New concise wording, Luna, medium reasoning", "openai/gpt-6-luna", "medium", short_wording_v2),
    ("Full wording, needed sections, Luna, medium reasoning", "openai/gpt-6-luna", "medium", sections_for(policy_wording)),
    ("New concise wording, needed sections, Luna, medium reasoning", "openai/gpt-6-luna", "medium", sections_for(short_wording_v2)),
]

diving = next(claim for claim in claims if claim["claim_id"] == "CLM-1217")
for name, *_, wording in SHORTENING_RUNS:
    text = wording(diving) if callable(wording) else wording
    headings = [line.removeprefix("## ") for line in text.splitlines() if line.startswith("## ")]
    print(f"{name}: {len(text.split()):,} words for CLM-1217, sections {', '.join(heading.split('.')[0] for heading in headings)}")

In [ ]:
for name, model_id, reasoning_effort, wording in SHORTENING_RUNS:
    if is_saved(name):
        print(f"{name}: already in {RESULTS}")
        continue
    save_results(name, run_model(name, model_id, reasoning_effort, wording))

The first two rows are experiment 4's runs on the same model: the full wording, and Ed's concise wording.

In [ ]:
shortening_names = ["Full wording, Luna, medium reasoning", "Concise wording, Luna, medium reasoning", *(name for name, *_ in SHORTENING_RUNS)]
shortening_tables = {name: table for name, table in saved_results().groupby("experiment") if name in shortening_names}
shortening_tables = {name: shortening_tables[name] for name in shortening_names if name in shortening_tables}

shortening_comparison = pd.DataFrame([average_scores(table) for table in shortening_tables.values()], index=list(shortening_tables))
shortening_comparison["seconds_per_claim"] = [table["seconds_per_claim"].mean() for table in shortening_tables.values()]
shortening_comparison.round(3)

The three claims where a lost meaning showed up in experiment 4, and CLM-1213, where every run with reasoning disagrees with the handler. Each cell is the agent's answer in the first run, marked when it's wrong.

In [ ]:
WATCHED_CLAIMS = ["CLM-1207", "CLM-1209", "CLM-1213", "CLM-1217"]


def answer_on(table, claim_id):
    """The agent's decision and payout on one claim in the first run, marked when it differs from the handler's."""
    row = table[(table["run"] == 1) & (table["claim_id"] == claim_id)].iloc[0]
    return f"{row['agent']} ${row['agent_payout']:,.2f}" + ("" if row["right"] else " (wrong)")


watched = {"Handler": {claim_id: f"{labels[claim_id]['decision']} ${labels[claim_id]['payout']:,.2f}" for claim_id in WATCHED_CLAIMS}}
watched |= {name: {claim_id: answer_on(table, claim_id) for claim_id in WATCHED_CLAIMS} for name, table in shortening_tables.items()}
pd.DataFrame(watched).T

In [ ]:
for name, table in shortening_tables.items():
    wrong = table[~table["right"]]
    print(f"{name}: {len(wrong)} wrong of {len(table)}")
    display(wrong[["run", "claim_id", "handler", "agent", "handler_payout", "agent_payout", "rationale"]])

### Conclusions on shortening

One run of each, on Luna with medium reasoning, on the 25 harder claims, on October 29, 2026. The first two rows are from experiment 4.

| Wording | Words per claim | Wrong of 25 | Cost of mistakes | Model cost per run |
|---|---|---|---|---|
| Full wording | 1,369 | 1 | $50 | $0.017 |
| Ed's concise wording | 808 | 4 | $1,928 | $0.017 |
| New concise wording | 1,191 | 1 | $50 | $0.016 |
| Full wording, needed sections | 670 to 860 | 0 | $0 | $0.017 |
| New concise wording, needed sections | 585 to 760 | 1 | $89 | $0.016 |

**The new concise wording fixes every meaning Ed's version lost.** CLM-1209, the grade 3 rafting, and CLM-1217, the dive to 9 meters, are both approved, as the handler decided, and CLM-1207 is paid to the cent. Its one mistake is a referral of CLM-1218, the hurricane, which no run with reasoning on any wording got wrong before, so it reads as the noise of one run.

**But a shortening that keeps the meaning has little to cut.** Kept word for word, the definitions, thresholds, lists and examples are most of the wording, so the new version is only 13% shorter than the full one. Ed's version was 41% shorter because it rewrote exactly those parts.

**Only the sections a claim needs is the best of the three, and the first run of Luna with no mistakes.** It changes no word of Ed's wording, so no meaning can be lost, and it cuts 40% to 50% of the wording for each claim. Section 8, the add-ons, has to go into every claim, because it holds the definitions of adventure sports and electronics: Week 1 Day 4 gave it only to policies with an add-on, which would have cost CLM-1209 and CLM-1217 here.

**Both together add little.** The new concise wording, in only the sections a claim needs, is the shortest prompt, and it got one payout wrong, on CLM-1215, by $89.

**CLM-1213, the bag on the beach, was decided right in all three runs.** Every one of the four earlier runs of Luna with reasoning on the full wording referred it. One run each can't say whether a shorter prompt makes the agent surer of "out of sight and out of reach", or whether these three runs were lucky. It is the first thing to check with more runs.

**The model's cost hardly moves, because the wording is already cached.** Every run costs 1.6 to 1.7 cents. OpenRouter and the provider behind Luna keep the start of the prompt that repeats from call to call, the tools, the instructions and the wording, and charge about a tenth of the price for it the next time. Each claim is still decided once: what is read again is the shared start of every prompt. In a check on 5 claims, 82% to 96% of the input tokens came from the cache, whichever wording was used, so the words a shorter wording cuts were mostly cheap already. At medium reasoning, most of what Luna charges is for the tokens it writes, its reasoning and its answer. On Luna, the case for a shorter wording is a clearer prompt, not a cheaper one.

**On these runs I would use the full wording, in only the sections a claim needs, on Luna with medium reasoning.** Before UnderCover relies on it, it should run three times (`N_TIMES = 3`) beside the full wording, since one run can't separate a mistake from noise.

## What I found

All on the 25 harder claims, one run of each, on October 29, 2026.

1. **The payout tool is the biggest single gain.** It takes Luna without reasoning from 14 mistakes to 3. Every change after it moves the result by a few mistakes at most.
2. **Reasoning is the cheapest next step, and the model matters less than it looks.** Luna at medium or high reasoning costs about 2 cents a run, against 70 to 85 cents for Grok 4.6 at the same levels, and makes 1 or 2 mistakes against Grok's 0. Luna at medium and at high ran twice each, and swapped places between experiments 3 and 4, with 2 and 1 mistakes at medium and 1 and 2 at high. That is noise, so I would take medium, which is as good and faster.
3. **Low reasoning confuses "needs sign-off" with "refer".** All six mistakes of Grok 4.6 and 4.7 at low reasoning are approvals over $1,000 that the agent worked out to the cent and then referred, because the payout tool said `needs_sign_off: true`. The course says an approval over $1,000 is held for sign-off, not referred, but the lab's instructions don't say so. One line in the instructions would be the next thing to try, before a bigger model.
4. **Ed's concise wording loses meaning in a few places, and they cost decisions.** "scuba below 10 meters" for "deeper than 10 meters" led to wrongful declines of a dive to 9 meters at two reasoning levels, and "Adventure Sports covers" for "Adventure sports are" left grade 3 rafting unclear. With reasoning on, the concise wording made 3 more mistakes than the full one at both medium and high, and saved nothing.
5. **Shorten by choosing, not by rewriting.** A rewrite that keeps every definition, threshold and example word for word fixed those mistakes, but cut only 13%. Giving each claim only the sections of the full wording it needs cut 40% to 50% without changing a word, and was the one run of Luna with no mistakes.
6. **Shortening saves accuracy, not money.** The wording is mostly read from the provider's cache at about a tenth of the price, so every wording costs Luna 1.6 to 1.7 cents a run. The reason to shorten is a prompt with nothing in it a claim doesn't need.
7. **CLM-1213 is the open question.** Luna with reasoning referred the bag left on the beach in all four runs on the full wording and both runs on Ed's concise one, and decided it right in all three runs of experiment 5. Three runs of each with `N_TIMES = 3` would show whether that is the shorter prompt or chance.

## Cleaning up Langfuse

This last step is optional. Set `CLEAN_UP = True` and run the cell to delete every trace in the Langfuse project your keys belong to, along with the experiments built on them, and the claims in the `harder-claims` dataset. This can't be undone. Only do it in a project you made for this course.

The empty `harder-claims` dataset stays, and running Part 4 again fills it. The run scores (accuracy, kappa and the rest) stay too. Langfuse's free plan allows 50 deletions of trace data a day, and each score would take one.

In [ ]:
def all_trace_ids():
    """Every trace id in the Langfuse project, collected from its observations a page at a time."""
    trace_ids = set()
    cursor = None
    while True:
        page = langfuse.api.observations.get_many(fields="core", limit=1000, cursor=cursor)
        trace_ids.update(observation.trace_id for observation in page.data)
        cursor = page.meta.cursor
        if not cursor:
            return list(trace_ids)


CLEAN_UP = False

if CLEAN_UP:
    trace_ids = all_trace_ids()
    for start in range(0, len(trace_ids), 1000):
        langfuse.api.trace.delete_multiple(trace_ids=trace_ids[start:start + 1000])

    dataset_items = langfuse.get_dataset(DATASET).items
    for item in dataset_items:
        langfuse.api.dataset_items.delete(id=item.id)

    print(f"Deleted {len(trace_ids)} traces and {len(dataset_items)} dataset items")